# Data summary (read-only)

Reads the JSON files in `../data/` and prints counts, **usability** (total vs usable),
keyword coverage, and per-source breakdowns (ECHR, RIS, Swiss — country/canton/court and
yearly stats). No plots, no files written, nothing modified. Defensive throughout (`.get`,
never assumes a key); reports expected-but-missing fields.

## 1. Inspect every JSON file in data/
Filename · top-level type · record count · field names (and whether fields are consistent
across records).

In [ ]:
import json, glob, os, re
from collections import Counter

DATA_DIR = "../data"


def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, str(e)


print("=" * 72)
print("JSON FILES IN", os.path.abspath(DATA_DIR))
print("=" * 72)
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.json"))):
    name = os.path.basename(path)
    data, err = load_json(path)
    if err:
        print(f"\n{name}\n   ERROR loading: {err}")
        continue
    if isinstance(data, list):
        dict_recs = [r for r in data if isinstance(r, dict)]
        print(f"\n{name}\n   top-level: list | records: {len(data)}")
        if dict_recs:
            keysets = [set(r.keys()) for r in dict_recs]
            union = set().union(*keysets)
            common = set.intersection(*keysets)
            print(f"   fields: {len(union)} | consistent across records: {union == common}")
            print(f"     {sorted(union)}")
            if union != common:
                print(f"     NOT on every record: {sorted(union - common)}")
        else:
            print("   (empty or no dict records)")
    elif isinstance(data, dict):
        print(f"\n{name}\n   top-level: dict | keys: {sorted(data.keys())}")
    else:
        print(f"\n{name}\n   top-level: {type(data).__name__}")

JSON FILES IN /Users/maksimsmirnov/Desktop/thesis/data

derstandard_comments.json
   top-level: list | records: 0
   (empty or no dict records)

derstandard_scrape_log.json
   top-level: list | records: 6
   fields: 6 | consistent across records: True
     ['article_id', 'article_url', 'error', 'n_comments', 'ok', 'scraped_at']

echr_domain_shift_llm_checkpoint.json
   top-level: dict | keys: ['001-101785', '001-102918', '001-104623', '001-104972', '001-106441', '001-106976', '001-110482', '001-111123', '001-111688', '001-113583', '001-114740', '001-116409', '001-119132', '001-123827', '001-126538', '001-128009', '001-138900', '001-139963', '001-142012', '001-142675', '001-145894', '001-147486', '001-148353', '001-21924', '001-23809', '001-5351', '001-59717', '001-59718', '001-61054', '001-61619', '001-67751', '001-70957', '001-71360', '001-75936', '001-81195', '001-82560', '001-82782', '001-83019', '001-85417', '001-99616']

echr_llm_extraction_checkpoint.json
   top-level: dict | key

## 2. ECHR — usability, keywords, breakdowns
Usable = non-empty `full_text` (whitespace stripped). All breakdowns are on the **usable** set.

In [ ]:
ECHR_PATH = os.path.join(DATA_DIR, "echr_parental_alienation.json")
echr, err = load_json(ECHR_PATH)
if err or not isinstance(echr, list):
    print(f"ECHR file unavailable ({err or 'not a list'}) — skipping ECHR summary")
    echr = []

# report expected-but-missing fields rather than crashing later
EXPECTED = ["full_text", "matched_keywords", "respondent", "judgementdate",
            "itemid", "appno", "ecli", "referencedate"]
if echr:
    seen = set().union(*[set(r.keys()) for r in echr[:100] if isinstance(r, dict)])
    missing = [f for f in EXPECTED if f not in seen]
    if missing:
        print("NOTE: expected ECHR fields not present:", missing, "\n")

total = len(echr)
usable = [r for r in echr if (r.get("full_text") or "").strip()]
print("=" * 72)
print("ECHR SUMMARY")
print("=" * 72)
print("USABILITY")
print(f"  total records         : {total}")
print(f"  usable (full_text)    : {len(usable)}")
print(f"  unusable (empty text) : {total - len(usable)}")

# keyword coverage on usable set
kw_present = sum(1 for r in usable if r.get("matched_keywords"))
kwc = Counter(k for r in usable for k in (r.get("matched_keywords") or []))
print("\nKEYWORDS (usable set)")
print(f"  usable with non-empty matched_keywords: {kw_present}/{len(usable)}")
for k, c in kwc.most_common():
    print(f"     {c:5d}  {k}")

# cases per respondent country (usable), descending
print("\nCASES PER RESPONDENT COUNTRY (usable, desc)")
for country, c in Counter((r.get("respondent") or "(none)") for r in usable).most_common():
    print(f"     {c:5d}  {country}")


def echr_year(r):
    # primary: judgementdate timestamp "DD/MM/YYYY 00:00:00"
    raw = (r.get("judgementdate") or "").strip()
    if raw:
        parts = raw.split("/")
        if len(parts) >= 3:
            try:
                return int(parts[2].split()[0][:4])
            except ValueError:
                pass
    # fallback date field: ECLI encodes the year (ECLI:CE:ECHR:YYYY:...)
    m = re.match(r"ECLI:CE:ECHR:(\d{4}):", r.get("ecli", "") or "")
    if m:
        return int(m.group(1))
    # fallback date field: referencedate (if populated)
    rd = (r.get("referencedate") or "").strip()
    if rd:
        mm = re.search(r"(\d{4})", rd)
        if mm:
            return int(mm.group(1))
    return None


years, noyear = Counter(), 0
for r in usable:
    y = echr_year(r)
    if y is None:
        noyear += 1
    else:
        years[y] += 1
print("\nCASES PER YEAR (usable; judgementdate, else ECLI/referencedate)")
for y in sorted(years):
    print(f"     {y}: {years[y]}")
print(f"     no parseable year: {noyear}")

# duplicate / inflation check
ids = [r.get("itemid") for r in echr if r.get("itemid")]
appnos = [r.get("appno") for r in echr if r.get("appno")]
print("\nDISTINCT-vs-RAW (is anything inflating totals?)")
print(f"  raw records      : {total}")
print(f"  distinct itemid  : {len(set(ids))}   (records carrying itemid: {len(ids)})")
print(f"  distinct appno   : {len(set(appnos))}   (records carrying appno: {len(appnos)})")

ECHR SUMMARY
USABILITY
  total records         : 1116
  usable (full_text)    : 1116
  unusable (empty text) : 0

KEYWORDS (usable set)
  usable with non-empty matched_keywords: 1116/1116
       748  best interests of the child
       554  contact rights
       227  child welfare
        41  parental alienation

CASES PER RESPONDENT COUNTRY (usable, desc)
       116  NOR
        99  RUS
        76  DEU
        73  POL
        67  ROU
        53  FIN
        49  GBR
        44  UKR
        43  HRV
        42  HUN
        41  BGR
        38  SWE
        29  AUT
        26  SRB
        24  CZE
        20  NLD
        18  SVK
        18  SVN
        18  LTU
        18  GRC
        17  FRA
        17  ITA
        17  LVA
        16  CHE
        15  PRT
        13  ESP
        11  TUR
        10  MDA
        10  EST
        10  MKD
         9  MLT
         8  DNK
         6  GEO
         6  ARM
         5  BEL
         5  AZE
         5  CYP
         4  ISL
         3  MNE
         3  IRL
  

## 3. RIS (Austria) — composition, keywords, yearly stats
Composition by document type (Rechtssatz principles vs Text full decisions — both used
downstream), keyword coverage, and cases per year from `entscheidungsdatum`.

In [ ]:
RIS_PATH = os.path.join(DATA_DIR, "ris_parental_alienation.json")
ris, err = load_json(RIS_PATH)
if err or not isinstance(ris, list):
    print(f"RIS file unavailable ({err or 'not a list'}) — skipping RIS summary")
    ris = []

if ris:
    seen = set().union(*[set(r.keys()) for r in ris if isinstance(r, dict)])
    for f in ["full_text", "matched_keywords", "id", "dokumenttyp", "entscheidungsdatum"]:
        if f not in seen:
            print("NOTE: expected RIS field not present:", f)


def ris_principle(full_text):
    # text between the 'Rechtssatz' label line and the 'Entscheidungstexte' label line
    lines = (full_text or "").split("\n")

    def find(label):
        for i, l in enumerate(lines):
            if l.strip() == label:
                return i
        return -1

    i_rs = find("Rechtssatz")
    if i_rs == -1:
        return ""
    i_et = find("Entscheidungstexte")
    i_ecli = find("European Case Law Identifier")
    end = i_et if i_et != -1 else (i_ecli if i_ecli != -1 else len(lines))
    return "\n".join(lines[i_rs + 1:end]).strip()


total = len(ris)
principled = [r for r in ris if ris_principle(r.get("full_text", "")).strip()]
decisions = [r for r in ris if not ris_principle(r.get("full_text", "")).strip()]
print("=" * 72)
print("RIS SUMMARY")
print("=" * 72)
print("COMPOSITION (by document type)")
print(f"  total records                 : {total}")
print(f"  Rechtssaetze (distilled principle) : {len(principled)}")
print(f"  Text (full decisions)              : {len(decisions)}")
print("  note: both are USED downstream — Rechtssaetze feed the 'principle' retrieval genre,")
print("        Text decisions the 'decision' genre. (Earlier versions indexed principles only.)")

kw_present = sum(1 for r in ris if r.get("matched_keywords"))
kwc = Counter(k for r in ris for k in (r.get("matched_keywords") or []))
print("\nKEYWORDS (all records)")
print(f"  with non-empty matched_keywords: {kw_present}/{total}")
for k, c in kwc.most_common():
    print(f"     {c:5d}  {k}")


def ris_year(r):
    m = re.match(r"(\d{4})", (r.get("entscheidungsdatum") or "").strip())
    return int(m.group(1)) if m else None


ris_years, ris_noyear = Counter(), 0
ris_years_rs, ris_years_txt = Counter(), Counter()
for r in ris:
    y = ris_year(r)
    if y is None:
        ris_noyear += 1
        continue
    ris_years[y] += 1
    (ris_years_rs if ris_principle(r.get("full_text", "")).strip() else ris_years_txt)[y] += 1
print("\nCASES PER YEAR (entscheidungsdatum; Text=decision date, Rechtssatz=latest applying decision)")
print(f"  {'year':>4} {'total':>6} {'Text':>6} {'Rechtssatz':>11}")
for y in sorted(ris_years):
    print(f"  {y:>4} {ris_years[y]:>6} {ris_years_txt.get(y, 0):>6} {ris_years_rs.get(y, 0):>11}")
print(f"  no parseable year: {ris_noyear}")
print("  caveat: a Rechtssatz's date is the most recent decision applying it (not its birth),")
print("          so only the 'Text' column is a clean decision-date series (see diachronic notebook).")

# duplicate / inflation check
ids = [r.get("id") for r in ris if r.get("id")]
print("\nDISTINCT-vs-RAW")
print(f"  raw records    : {total}")
print(f"  distinct id    : {len(set(ids))}   (records carrying id: {len(ids)})")

RIS SUMMARY
COMPOSITION (by document type)
  total records                 : 548
  Rechtssaetze (distilled principle) : 38
  Text (full decisions)              : 510
  note: both are USED downstream — Rechtssaetze feed the 'principle' retrieval genre,
        Text decisions the 'decision' genre. (Earlier versions indexed principles only.)

KEYWORDS (all records)
  with non-empty matched_keywords: 548/548
       394  Kindeswohlgefährdung
       159  Entfremdung
         4  Loyalitätskonflikt

CASES PER YEAR (entscheidungsdatum; Text=decision date, Rechtssatz=latest applying decision)
  year  total   Text  Rechtssatz
  2000     14     14           0
  2001     17     17           0
  2002     17     17           0
  2003     20     20           0
  2004     12     12           0
  2005     14     14           0
  2006     21     20           1
  2007     14     14           0
  2008     14     14           0
  2009     20     20           0
  2010     32     31           1
  2011     22 

## 4. Swiss (entscheidsuche.ch) — usability, keywords, canton/court/year breakdowns
Usable = non-empty `content` (the ES-extracted decision text; Swiss records carry text in
`content`, not `full_text`). Breakdowns on the usable set: matched keywords, cases per
canton, per court type, and per year (`year`, else `Datum`).

In [ ]:
SWISS_PATH = os.path.join(DATA_DIR, "swiss_parental_alienation.json")
swiss, err = load_json(SWISS_PATH)
if err or not isinstance(swiss, list):
    print(f"Swiss file unavailable ({err or 'not a list'}) — skipping Swiss summary")
    swiss = []

if swiss:
    seen = set().union(*[set(r.keys()) for r in swiss[:100] if isinstance(r, dict)])
    for f in ["content", "matched_keywords", "canton", "court_type", "year", "stable_id"]:
        if f not in seen:
            print("NOTE: expected Swiss field not present:", f)

total = len(swiss)
usable = [r for r in swiss if (r.get("content") or "").strip()]
print("=" * 72)
print("SWISS SUMMARY")
print("=" * 72)
print("USABILITY")
print(f"  total records         : {total}")
print(f"  usable (content)      : {len(usable)}")
print(f"  unusable (empty text) : {total - len(usable)}")

kw_present = sum(1 for r in usable if r.get("matched_keywords"))
kwc = Counter(k for r in usable for k in (r.get("matched_keywords") or []))
print("\nKEYWORDS (usable set)")
print(f"  usable with non-empty matched_keywords: {kw_present}/{len(usable)}")
for k, c in kwc.most_common():
    print(f"     {c:5d}  {k}")

print("\nCASES PER CANTON (usable, desc)")
for canton, c in Counter((r.get("canton") or "(none)") for r in usable).most_common():
    print(f"     {c:5d}  {canton}")

print("\nCASES PER COURT TYPE (usable, top 15)")
for ct, c in Counter((r.get("court_type") or "(none)") for r in usable).most_common(15):
    print(f"     {c:5d}  {ct}")


def swiss_year(r):
    y = r.get("year")
    if isinstance(y, int):
        return y
    m = re.match(r"(\d{4})", str(y or r.get("Datum") or "").strip())
    return int(m.group(1)) if m else None


sw_years, sw_noyear = Counter(), 0
for r in usable:
    y = swiss_year(r)
    if y is None:
        sw_noyear += 1
    else:
        sw_years[y] += 1
print("\nCASES PER YEAR (usable; year field, else Datum)")
for y in sorted(sw_years):
    print(f"     {y}: {sw_years[y]}")
print(f"     no parseable year: {sw_noyear}")

ids = [r.get("stable_id") for r in swiss if r.get("stable_id")]
print("\nDISTINCT-vs-RAW")
print(f"  raw records       : {total}")
print(f"  distinct stable_id: {len(set(ids))}   (records carrying stable_id: {len(ids)})")

SWISS SUMMARY
USABILITY
  total records         : 2031
  usable (content)      : 2031
  unusable (empty text) : 0

KEYWORDS (usable set)
  usable with non-empty matched_keywords: 2031/2031
      1002  Loyalitätskonflikt
       819  Kindeswohlgefährdung
       565  Entfremdung
       126  Kontaktverweigerung
        14  Eltern-Kind-Entfremdung
         1  elterliche Entfremdung

CASES PER CANTON (usable, desc)
       621  ZH
       514  CH
       177  BL
       172  GR
       136  BS
       123  AG
        57  BE
        45  SO
        36  FR
        24  TG
        23  SG
        16  VS
        15  NW
        15  LU
        14  SZ
        14  ZG
        13  AR
        10  SH
         2  UR
         2  GL
         1  AI
         1  TA

CASES PER COURT TYPE (usable, top 15)
       608  ZH_OG
       491  CH_BGer
       176  BL_KG
       172  GR_KG
       136  BS_APG
       120  AG_OG
        52  BE_OG
        45  SO_OG
        36  FR_TC
        24  TG_OG
        23  SG_KG
        20  CH_BG

## Notes
- ECHR usable = non-empty `full_text`; RIS usable = non-empty extracted principle (the
  `Text` full-decision records have no principle layout, so they count as unusable here).
- Read-only: no data file or other notebook was modified, and nothing was written to disk.